# LAB 1: THU THẬP DỮ LIỆU WEB (WEB SCRAPING)
**Mã số SV**: 26410022  
**Mục tiêu**:
1. Sử dụng thư viện `requests` để gửi HTTP request lấy nội dung trang web.
2. Dùng `BeautifulSoup` để parse cây DOM và bóc tách dữ liệu theo thẻ/class.
3. Thu thập dữ liệu chi tiết bài viết (tiêu đề, mô tả tóm tắt, URL).
4. Lưu dữ liệu thu thập được thành file định dạng CSV (`.csv`).
5. Thực hành mở rộng với chuyên mục khác.


## 1. Cài đặt và import các thư viện cần thiết

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time

print("Imported successfully!")


Imported successfully!


## 2. Gửi HTTP Request và kiểm tra phản hồi
Lấy trang chuyên mục bóng đá của VnExpress: `https://vnexpress.net/bong-da`

In [2]:
url = "https://vnexpress.net/bong-da"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("HTTP Status Code:", response.status_code)


HTTP Status Code: 200


## 3. Tách dữ liệu bằng BeautifulSoup và trích xuất danh sách link bài viết
Tìm tất cả thẻ `h2` / `h3` có class `title-news` chứa link bài viết:

In [3]:
soup = BeautifulSoup(response.content, "html.parser")
titles = soup.find_all(["h2", "h3"], class_="title-news")

links = []
for item in titles:
    a_tag = item.find("a")
    if a_tag and a_tag.get("href"):
        link = a_tag["href"]
        if link.startswith("https://vnexpress.net/") and link.endswith(".html"):
            if link not in links:
                links.append(link)

print(f"Tổng số link bài viết lấy được: {len(links)}")
for l in links[:5]:
    print("-", l)


Tổng số link bài viết lấy được: 30
- https://vnexpress.net/mbappe-that-kech-kom-neu-than-van-ve-viec-lam-mbappe-kho-the-nao-5137151.html
- https://vnexpress.net/qua-bong-vang-va-cuoc-dua-khoc-liet-ca-trong-lan-ngoai-san-co-5136596.html
- https://vnexpress.net/muu-luoc-cua-mourinho-o-el-clasico-5136955.html
- https://vnexpress.net/gia-toc-500-nam-tuoi-nam-giu-bi-mat-qua-bong-vang-5136723.html
- https://vnexpress.net/dang-sau-cau-trich-dan-cua-cong-chua-tay-ban-nha-ve-messi-5136374.html


## 4. Lấy chi tiết từng bài viết (Tiêu đề, Mô tả)

In [4]:
articles_data = []

for idx, link in enumerate(links[:10], 1):
    try:
        news_res = requests.get(link, headers=headers, timeout=10)
        if news_res.status_code != 200:
            continue
        
        news_soup = BeautifulSoup(news_res.content, "html.parser")
        
        # Tiêu đề bài viết
        title_tag = news_soup.find("h1", class_="title-detail") or news_soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else "N/A"
        
        # Mô tả / Tóm tắt bài viết
        desc_tag = news_soup.find("p", class_="description") or news_soup.find("p")
        desc = desc_tag.get_text(strip=True) if desc_tag else "N/A"
        
        articles_data.append({
            "STT": idx,
            "Tiêu đề": title,
            "Mô tả": desc,
            "Đường dẫn": link
        })
        
        print(f"[{idx}] Tiêu đề: {title}")
        print(f"    Mô tả  : {desc[:80]}...")
        print("-" * 50)
        time.sleep(0.3)
    except Exception as e:
        print(f"Lỗi: {e}")


[1] Tiêu đề: Mbappe: 'Thật kệch kợm nếu than vãn về việc làm Mbappe khổ thế nào'
    Mô tả  : Kylian Mbappe trong cuộc phỏng vấn với tạp chí Pháp Onze Mondial ngày 20/10/2026...
--------------------------------------------------


[2] Tiêu đề: Quả Bóng Vàng và cuộc đua khốc liệt cả trong lẫn ngoài sân cỏ
    Mô tả  : Tháng 9/2025, khi Ousmane Dembele còn chưa kịp nguôi xúc động sau khoảnh khắc nh...
--------------------------------------------------


[3] Tiêu đề: Mưu lược của Mourinho ở El Clasico
    Mô tả  : Theo báo Tây Ban Nha AS, cách tiếp cận trận đấu và các phương án nhân sự của HLV...
--------------------------------------------------


[4] Tiêu đề: Gia tộc 500 năm tuổi nắm giữ bí mật Quả Bóng Vàng
    Mô tả  : Gia đình Mellerio, chủ sở hữu thương hiệu kim hoàn lâu đời nhất thế giới, nắm gi...
--------------------------------------------------


[5] Tiêu đề: Đằng sau câu trích dẫn của Công chúa Tây Ban Nha về Messi
    Mô tả  : Cựu cầu thủ Sporting Gijon được Công chúa Tây Ban Nha trích dẫn câu nói về Lione...
--------------------------------------------------


[6] Tiêu đề: Ronaldo vượt Messi ở bảng điểm lịch sử Quả Bóng Vàng
    Mô tả  : Cristiano Ronaldo dẫn Lionel Messi 460-450 điểm trong bảng thống kê lịch sử Quả ...
--------------------------------------------------


[7] Tiêu đề: Lewandowski ủng hộ Yamal giành Quả Bóng Vàng 2026
    Mô tả  : Tây Ban NhaTheo cựu tiền đạo Barca Robert Lewandowski, với những gì đã thể hiện,...
--------------------------------------------------


[8] Tiêu đề: Mbappe chạm mốc 100 bàn cho Real Madrid
    Mô tả  : Với cú đúp hạ Barca 2-1 ở El Clasico ngày 25/10, tiền đạo Kylian Mbappe có tròn ...
--------------------------------------------------


[9] Tiêu đề: Báo Tây Ban Nha: 'Cucurella không cho Yamal thở ở El Clasico'
    Mô tả  : Hậu vệ Marc Cucurella nhận nhiều lời ngợi khen khi vô hiệu hóa Lamine Yamal, góp...
--------------------------------------------------


[10] Tiêu đề: Mourinho: 'Real xứng đáng thắng Barca'
    Mô tả  : Tây Ban NhaHLV Jose Mourinho khen ngợi nhiều cầu thủ, đồng thời chỉ trích VAR "n...
--------------------------------------------------


## 5. Lưu dữ liệu thu thập được thành file CSV

In [5]:
from pathlib import Path
output_dir = Path("datasets/lab1")
output_dir.mkdir(parents=True, exist_ok=True)
csv_filename = output_dir / "vnexpress_bongda.csv"

# Xuất ra file CSV (chuẩn UTF-8 with BOM để mở Excel tiếng Việt không bị lỗi)
with open(csv_filename, mode="w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["STT", "Tiêu đề", "Mô tả", "Đường dẫn"])
    writer.writeheader()
    writer.writerows(articles_data)

print(f"Đã lưu thành công {len(articles_data)} dòng dữ liệu vào {csv_filename}")

# Đọc lại và hiển thị bảng dữ liệu bằng pandas
df = pd.read_csv(csv_filename)
df.head()


Đã lưu thành công 10 dòng dữ liệu vào vnexpress_bongda.csv


,STT,Tiêu đề,Mô tả,Đường dẫn
0,1,Mbappe: 'Thật kệch kợm nếu than vãn về việc là...,Kylian Mbappe trong cuộc phỏng vấn với tạp chí...,https://vnexpress.net/mbappe-that-kech-kom-neu...
1,2,Quả Bóng Vàng và cuộc đua khốc liệt cả trong l...,"Tháng 9/2025, khi Ousmane Dembele còn chưa kịp...",https://vnexpress.net/qua-bong-vang-va-cuoc-du...
2,3,Mưu lược của Mourinho ở El Clasico,"Theo báo Tây Ban Nha AS, cách tiếp cận trận đấ...",https://vnexpress.net/muu-luoc-cua-mourinho-o-...
3,4,Gia tộc 500 năm tuổi nắm giữ bí mật Quả Bóng Vàng,"Gia đình Mellerio, chủ sở hữu thương hiệu kim ...",https://vnexpress.net/gia-toc-500-nam-tuoi-nam...
4,5,Đằng sau câu trích dẫn của Công chúa Tây Ban N...,Cựu cầu thủ Sporting Gijon được Công chúa Tây ...,https://vnexpress.net/dang-sau-cau-trich-dan-c...


## 6. Mở rộng: Hàm crawl đa năng áp dụng cho danh mục khác

In [6]:
from pathlib import Path

def crawl_and_save(category_url, output_csv, limit=10):
    print(f"Bắt đầu crawl từ: {category_url}")
    res = requests.get(category_url, headers=headers, timeout=10)
    sp = BeautifulSoup(res.content, "html.parser")
    items = sp.find_all(["h2", "h3"], class_="title-news")
    
    target_links = []
    for it in items:
        a = it.find("a")
        if a and a.get("href"):
            h = a["href"]
            if h.startswith("https://vnexpress.net/") and h.endswith(".html") and h not in target_links:
                target_links.append(h)
        if len(target_links) >= limit:
            break
            
    results = []
    for i, lnk in enumerate(target_links, 1):
        r = requests.get(lnk, headers=headers, timeout=10)
        s = BeautifulSoup(r.content, "html.parser")
        t = s.find("h1", class_="title-detail") or s.find("h1")
        d = s.find("p", class_="description") or s.find("p")
        
        results.append({
            "STT": i,
            "Tiêu đề": t.get_text(strip=True) if t else "N/A",
            "Mô tả": d.get_text(strip=True) if d else "N/A",
            "Đường dẫn": lnk
        })
        time.sleep(0.3)
        
    df_out = pd.DataFrame(results)
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"Đã hoàn thành và lưu vào {output_csv} ({len(df_out)} bài)")
    return df_out

# Thử nghiệm với chuyên mục Khoa học
df_khoahoc = crawl_and_save("https://vnexpress.net/khoa-hoc", "datasets/lab1/vnexpress_khoahoc.csv", limit=5)
df_khoahoc


Bắt đầu crawl từ: https://vnexpress.net/khoa-hoc


Đã hoàn thành và lưu vào vnexpress_khoahoc.csv (4 bài)


,STT,Tiêu đề,Mô tả,Đường dẫn
0,1,Kỹ sư Việt làm hệ thống tính cước viễn thông c...,Hệ thống tính cước vOCS 4.0 do kỹ sư Viettel H...,https://vnexpress.net/ky-su-viet-lam-he-thong-...
1,2,Bill Gates cảnh báo 20 năm biến động trước khi...,"Tỷ phú Bill Gates cho rằng AI sẽ mang đến ""kỷ ...",https://vnexpress.net/bill-gates-canh-bao-20-n...
2,3,"Hai Bộ trưởng Tài chính, Khoa học và Công nghệ...",Bộ trưởng Tài chính Ngô Văn Tuấn và Bộ trưởng ...,https://vnexpress.net/hai-bo-truong-tai-chinh-...
3,4,'AI giúp hacker khai thác lỗ hổng dưới một ngày',AI đang rút ngắn thời gian biến lỗ hổng bảo mậ...,https://vnexpress.net/ai-giup-hacker-khai-thac...
